# GR-Prediction TCN — Colab Training

Trains a non-parametric TCN to predict the gain-reduction envelope of the
SSL G-Bus compressor from dry audio input.

**Runtime**: Select **GPU** (T4 is fine) via *Runtime → Change runtime type*.

In [ ]:
# ── 0. Install dependencies ──────────────────────────────────────────
!pip install -q torch torchaudio lightning nablafx soundfile huggingface_hub

In [ ]:
# ── 1. Download dataset from Hugging Face ────────────────────────────
#
# The Diff-SSL-G-Comp dataset is gated — you need a HF token with access.
# Paste your token below, or set it via Colab Secrets (recommended).

import os
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = ""  # paste your token here if not using Colab Secrets

os.environ["HF_TOKEN"] = HF_TOKEN

SETTING = "threshold_-4_attack_1_release_0.4_ratio_10"

from huggingface_hub import snapshot_download

DATA_ROOT = snapshot_download(
    repo_id="amphion/Diff-SSL-G-Comp",
    repo_type="dataset",
    allow_patterns=[
        "processed_normalized/*.wav",
        f"processed_ground_truth/{SETTING}/*.wav",
    ],
    local_dir="/content/Diff-SSL-G-Comp",
    token=HF_TOKEN or None,
)
print(f"Dataset root: {DATA_ROOT}")

In [ ]:
# ── 2. Shared DSP utilities (inlined from src/) ──────────────────────

import torch
import torch.nn.functional as F

# --- constants ---
PARAM_ORDER = ["threshold", "attack", "release", "ratio"]
PARAM_RANGES_LOCAL = {
    "threshold": (-20.0, 0.0),
    "attack": (0.1, 30.0),
    "release": (0.1, 1.6),
    "ratio": (2.0, 10.0),
}

GR_DB_MIN = -30.0
GR_DB_MAX = 0.0
RMS_WINDOW = 1024


def windowed_rms(signal: torch.Tensor, window_size: int) -> torch.Tensor:
    sq = (signal.unsqueeze(0)) ** 2
    kernel = torch.ones(1, 1, window_size, device=signal.device) / window_size
    pad = window_size - 1
    rms_sq = F.conv1d(sq, kernel, padding=pad)
    rms_sq = rms_sq[..., : signal.shape[-1]]
    return torch.sqrt(rms_sq.squeeze(0).clamp(min=1e-10))


def gain_reduction_db(
    dry: torch.Tensor, wet: torch.Tensor, window_size: int = RMS_WINDOW
) -> torch.Tensor:
    dry_rms = windowed_rms(dry, window_size)
    wet_rms = windowed_rms(wet, window_size)
    dry_db = 20 * torch.log10(dry_rms)
    wet_db = 20 * torch.log10(wet_rms)
    return wet_db - dry_db


def normalize_gr(gr_db: torch.Tensor) -> torch.Tensor:
    return (gr_db - GR_DB_MIN) / (GR_DB_MAX - GR_DB_MIN) * 2 - 1


def denormalize_gr(gr_norm: torch.Tensor) -> torch.Tensor:
    return (gr_norm + 1) / 2 * (GR_DB_MAX - GR_DB_MIN) + GR_DB_MIN

In [ ]:
# ── 3. Dataset & DataModule ──────────────────────────────────────────

import glob
import soundfile as sf
import torchaudio
import lightning as pl
from torch.utils.data import Dataset, DataLoader
from typing import Optional

SAMPLE_RATE = 44100
SAMPLE_LENGTH = 132300  # 3 s


class GainReductionDataset(Dataset):
    def __init__(
        self,
        data_root: str,
        settings_folder: str,
        sample_length: int = SAMPLE_LENGTH,
        sample_rate: int = SAMPLE_RATE,
        rms_window: int = RMS_WINDOW,
    ):
        self.sample_length = sample_length
        self.sample_rate = sample_rate
        self.rms_window = rms_window

        dry_dir = os.path.join(data_root, "processed_normalized")
        wet_dir = os.path.join(data_root, "processed_ground_truth", settings_folder)

        if not os.path.isdir(dry_dir):
            raise FileNotFoundError(f"Dry directory not found: {dry_dir}")
        if not os.path.isdir(wet_dir):
            raise FileNotFoundError(f"Wet directory not found: {wet_dir}")

        dry_lookup: dict[str, str] = {}
        for p in sorted(glob.glob(os.path.join(dry_dir, "*_UnmasteredWAV.wav"))):
            song = os.path.basename(p).replace("_UnmasteredWAV.wav", "")
            dry_lookup[song] = p

        pairs: list[tuple[str, str]] = []
        for wet_path in sorted(glob.glob(os.path.join(wet_dir, "*-exported.wav"))):
            song = os.path.basename(wet_path).replace("-exported.wav", "")
            if song in dry_lookup:
                pairs.append((dry_lookup[song], wet_path))

        if not pairs:
            raise ValueError(f"No matching dry/wet pairs for setting '{settings_folder}'")

        self.samples: list[dict] = []
        for dry_path, wet_path in pairs:
            md = sf.info(dry_path)
            n_frames = md.frames
            if sample_length == -1:
                self.samples.append(
                    {"dry": dry_path, "wet": wet_path, "offset": 0, "frames": n_frames}
                )
            else:
                for n in range(n_frames // sample_length):
                    self.samples.append(
                        {
                            "dry": dry_path,
                            "wet": wet_path,
                            "offset": n * sample_length,
                            "frames": sample_length,
                        }
                    )

        print(
            f"GainReductionDataset: {len(pairs)} songs, "
            f"{len(self.samples)} chunks  "
            f"[setting={settings_folder}]"
        )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        s = self.samples[idx]
        nf = s["frames"] if self.sample_length != -1 else -1

        dry, sr = sf.read(
            s["dry"], start=s["offset"],
            stop=None if nf == -1 else s["offset"] + nf,
            dtype="float32", always_2d=True,
        )
        dry = torch.from_numpy(dry.T)

        wet, sr_w = sf.read(
            s["wet"], start=s["offset"],
            stop=None if nf == -1 else s["offset"] + nf,
            dtype="float32", always_2d=True,
        )
        wet = torch.from_numpy(wet.T)

        if sr != self.sample_rate:
            dry = torchaudio.functional.resample(dry, sr, self.sample_rate)
        if sr_w != self.sample_rate:
            wet = torchaudio.functional.resample(wet, sr_w, self.sample_rate)

        if dry.shape[0] > 1:
            dry = dry.mean(dim=0, keepdim=True)
        if wet.shape[0] > 1:
            wet = wet.mean(dim=0, keepdim=True)

        min_len = min(dry.shape[-1], wet.shape[-1])
        dry = dry[..., :min_len]
        wet = wet[..., :min_len]

        gr = gain_reduction_db(dry, wet, self.rms_window)
        gr = normalize_gr(gr).clamp(-1.0, 1.0)

        return dry, gr


class GainReductionDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_root: str,
        settings_folder: str,
        sample_length: int = SAMPLE_LENGTH,
        sample_rate: int = SAMPLE_RATE,
        rms_window: int = RMS_WINDOW,
        train_split: float = 0.8,
        batch_size: int = 16,
        num_workers: int = 2,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.data_root = data_root
        self.settings_folder = settings_folder
        self.sample_length = sample_length
        self.sample_rate = sample_rate
        self.rms_window = rms_window
        self.train_split = train_split
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage: Optional[str] = None) -> None:
        full = GainReductionDataset(
            data_root=self.data_root,
            settings_folder=self.settings_folder,
            sample_length=self.sample_length,
            sample_rate=self.sample_rate,
            rms_window=self.rms_window,
        )
        n_train = int(len(full) * self.train_split)
        n_val = len(full) - n_train
        self.train_dataset, self.val_dataset = torch.utils.data.random_split(
            full, [n_train, n_val]
        )
        print(f"Train: {n_train}  Val: {n_val}")

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            drop_last=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

In [ ]:
# ── 4. Lightning system ──────────────────────────────────────────────

import types, sys
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

if "rational" not in sys.modules:
    _r = types.ModuleType("rational")
    _rt = types.ModuleType("rational.torch")
    _rt.Rational = type("Rational", (torch.nn.Module,), {"forward": lambda self, x: x})
    _r.torch = _rt
    sys.modules["rational"] = _r
    sys.modules["rational.torch"] = _rt

from nablafx.processors import TCN


class GRPredictionSystem(pl.LightningModule):
    def __init__(
        self,
        processor: torch.nn.Module,
        lr: float = 1e-3,
        smooth_l1_beta: float = 0.5,
        diff_weight: float = 0.1,
        diff_beta: float = 0.1,
    ):
        super().__init__()
        self.processor = processor
        self.lr = lr
        self.smooth_l1_beta = smooth_l1_beta
        self.diff_weight = diff_weight
        self.diff_beta = diff_beta

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.processor(x)

    def _step(self, batch: tuple, mode: str) -> torch.Tensor:
        dry, gr_target = batch
        if hasattr(self.processor, "reset_states"):
            self.processor.reset_states()
        gr_pred = self(dry)

        main_loss = F.smooth_l1_loss(gr_pred, gr_target, beta=self.smooth_l1_beta)

        dp = gr_pred[..., 1:] - gr_pred[..., :-1]
        dt = gr_target[..., 1:] - gr_target[..., :-1]
        diff_loss = F.smooth_l1_loss(dp, dt, beta=self.diff_beta)

        loss = main_loss + self.diff_weight * diff_loss

        self.log(f"loss/{mode}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"loss/{mode}_main", main_loss, on_step=False, on_epoch=True)
        self.log(f"loss/{mode}_diff", diff_loss, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", patience=10, factor=0.5
        )
        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": sched,
                "monitor": "loss/val",
                "interval": "epoch",
            },
        }

In [ ]:
# ── 5. Train (10-minute run) ─────────────────────────────────────────

BATCH_SIZE = 32
LR = 3e-3
MAX_TIME = "00:00:10:00"  # DD:HH:MM:SS

dm = GainReductionDataModule(
    data_root=DATA_ROOT,
    settings_folder=SETTING,
    sample_length=SAMPLE_LENGTH,
    sample_rate=SAMPLE_RATE,
    rms_window=RMS_WINDOW,
    train_split=0.8,
    batch_size=BATCH_SIZE,
    num_workers=2,
)

tcn = TCN(
    num_inputs=1,
    num_outputs=1,
    num_controls=0,
    num_blocks=10,
    kernel_size=3,
    dilation_growth=2,
    channel_width=32,
    causal=True,
    cond_type=None,
    bias=False,
    batchnorm=True,
)
print(f"TCN receptive field: {tcn.rf} samples ({tcn.rf / SAMPLE_RATE:.3f} s)")

system = GRPredictionSystem(
    processor=tcn,
    lr=LR,
    smooth_l1_beta=0.5,
    diff_weight=0.1,
    diff_beta=0.1,
)

ckpt_cb = ModelCheckpoint(
    monitor="loss/val",
    mode="min",
    save_top_k=1,
    save_last=True,
    filename="best-{epoch}-{step}",
)
lr_cb = LearningRateMonitor(logging_interval="epoch")

trainer = pl.Trainer(
    max_epochs=100,
    max_time=MAX_TIME,
    accelerator="auto",
    precision="16-mixed",
    callbacks=[ckpt_cb, lr_cb],
    log_every_n_steps=10,
    default_root_dir="/content/lightning_logs",
)

trainer.fit(system, dm)

In [ ]:
# ── 6. Quick look at loss curve ──────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt

metrics_csv = sorted(glob.glob("/content/lightning_logs/lightning_logs/version_*/metrics.csv"))[-1]
df = pd.read_csv(metrics_csv)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df["epoch"].dropna(), df["loss/train"].dropna(), label="train")
ax.plot(df["epoch"].dropna(), df["loss/val"].dropna(), label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.set_title("GR Prediction Loss")
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Download checkpoint ───────────────────────────────────────────

from google.colab import files

best_ckpt = ckpt_cb.best_model_path
print(f"Best checkpoint: {best_ckpt}  (val loss: {ckpt_cb.best_model_score:.4f})")
files.download(best_ckpt)